# JMAIL Dataset: Esplorazione Raw e Validazione Preprocessing

Questo notebook documenta la verifica della pipeline di cleaning e feature engineering.
La logica di preprocessing è importata da `src.utils.data_processing`, garantendo modularità e riproducibilità.

## 1. Setup

Carichiamo librerie e configurazioni dal file `.env`.

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from dotenv import dotenv_values

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.data_processing import run_processing_with_limit

ENV_PATH = PROJECT_ROOT / ".env"
ENV = dotenv_values(ENV_PATH)

def project_path(env_key, default):
    value = Path(ENV.get(env_key, default))
    return value if value.is_absolute() else PROJECT_ROOT / value

PROCESSING_SAMPLE_SIZE = int(ENV.get("PROCESSING_SAMPLE_SIZE", "1000"))
SAMPLE_SIZE = PROCESSING_SAMPLE_SIZE if PROCESSING_SAMPLE_SIZE > 0 else 1000
PROCESSING_LIMIT = None if PROCESSING_SAMPLE_SIZE == -1 else PROCESSING_SAMPLE_SIZE
RAW_PATH = project_path("DATA_RAW_PATH", "data/raw/") / ENV.get("RAW_EMAILS_FILENAME", "jmail_emails.parquet")
PROCESSED_SAMPLE_PATH = project_path("DATA_PROCESSED_PATH", "data/processed/") / ENV.get(
    "PROCESSED_EMAILS_SAMPLE_FILENAME", "jmail_emails_processed_sample.parquet"
)
PROCESSING_METADATA_PATH = project_path("METADATA_PATH", "data/metadata/") / ENV.get(
    "PROCESSING_SAMPLE_METADATA_FILENAME", "jmail_processing_sample_metadata.json"
)
PROFILE_PATH = project_path("METADATA_PATH", "data/metadata/") / ENV.get("PROFILE_METADATA_FILENAME", "jmail_profile.json")

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

## 2. Caricamento Sample Raw

Osserviamo i dati prima del preprocessing.

In [2]:
parquet_file = pq.ParquetFile(RAW_PATH)
first_batch = next(parquet_file.iter_batches(batch_size=SAMPLE_SIZE))
raw_sample = pa.Table.from_batches([first_batch]).to_pandas()

print(f"Righe sample raw: {len(raw_sample):,}")
raw_sample.head(3)

Righe sample raw: 1,000


,id,doc_id,message_index,sender,subject,to_recipients,cc_recipients,bcc_recipients,sent_at,content_markdown,content_html,attachments,account_email,email_drop_id,folder_path,is_promotional,release_batch,epstein_is_sender,all_participants
0,000973eb49f43cdcead0876fb2de6989,9ed3cf0dc144d9ef7df792bb2fa74c52,0,Houzz Updates <updates@houzz.com>,Color of the Week | 1940s Ranch Redo | Your July Home Checklist,"[""<jeeproject@yahoo.com>""]",[],[],2015-07-01T13:26:22.000Z,"To-Dos: Your July Home Checklist\n\nCrank up the ice cream maker, hang up the hammock and raise the outdoor umbrella...","<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0 Transitional//EN"" ""http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional....",0,jeeproject@yahoo.com,yahoo_2,Inbox,True,1,False,"houzz updates <updates@houzz.com> [""<jeeproject@yahoo.com>""] [] []"
1,000f08c9cb90acbc5332fcbb1915f985,584bc87a2226599969664a61fd4e39fe,0,Spotify <hello@spotify.com>,"Upcoming shows near New York: Michael Feinstein, Boston Pops Orchestra and more","[""<jeeproject@yahoo.com>""]",[],[],2018-09-26T02:00:00.000Z,"Upcoming concerts near you by artists you love.\n\n *Michael Feinstein*\n\n Sun, Oct 28\n\n New Jersey ...","<!DOCTYPE html><html xmlns=""http://www.w3.org/1999/xhtml"" style=""margin: 0; padding: 0""><head><meta charset=""utf-8"" ...",0,jeeproject@yahoo.com,yahoo_2,Inbox,True,1,False,"spotify <hello@spotify.com> [""<jeeproject@yahoo.com>""] [] []"
2,000fa4d83ae293b30ed3571c9ea2fd77,905350832d97dc3675bfed84993cd746,0,jeffrey E. <jeevacation@gmail.com>,Re: Title and subtitle,"[""Ehud Barak <ehbarak1@gmail.com>""]",[],NaN,2015-05-07T07:07:33.000Z,"prefer OUR COUNTRY , MY LIFE or MOMENTS","<div dir=""ltr"">prefer OUR COUNTRY , MY LIFE or MOMENTS<br>",0,ehbarak1@gmail.com,ehud_ddos_dropsite_1,NaN,False,8,True,"jeffrey e. <jeevacation@gmail.com> [""ehud barak <ehbarak1@gmail.com>""] []"


## 3. Esecuzione Pipeline su Sample

Eseguiamo il preprocessing configurato (Phase 1).

In [3]:
processing_result = run_processing_with_limit(env_file=ENV_PATH, limit=PROCESSING_LIMIT)
processing_result

{'raw_path': '/home/suga/Workspace/Projects/pizza-cluster/data/raw/jmail_emails.parquet',
 'processed_output_path': '/home/suga/Workspace/Projects/pizza-cluster/data/processed/jmail_emails_processed_sample.parquet',
 'metadata_output_path': '/home/suga/Workspace/Projects/pizza-cluster/data/metadata/jmail_processing_sample_metadata.json',
 'input_rows': 1000,
 'output_rows': 355,
 'removed_promotional_rows': 645,
 'removed_empty_text_rows': 0}

## 4. Analisi Dati Processati

Verifichiamo le nuove feature (redaction_ratio, template, ecc.).

In [4]:
processed_sample = pd.read_parquet(PROCESSED_SAMPLE_PATH)
print(f"Processed rows: {len(processed_sample):,}")
display(processed_sample[["id", "combined_text", "redaction_count", "redaction_ratio"]].head(5))

Processed rows: 355


,id,combined_text,redaction_count,redaction_ratio
0,000fa4d83ae293b30ed3571c9ea2fd77,"DATE: 2015-05-07T07:07:33.000Z\nFROM: jeffrey E. <jeevacation@gmail.com>\nTO: [""Ehud Barak <ehbarak1@gmail.com>""]\...",0,0.0
1,001612df62eb14194162f0a366793927,"DATE: 2006-07-10T18:13:50.000Z\nFROM: J. Epstein <jeeproject@yahoo.com>\nTO: [""Cecilia Steen <cecilia.steen@gmail.co...",0,0.0
2,001df92b110e9da90631a66cf97a0a11,"DATE: 2007-02-19T20:50:39.000Z\nFROM: J. Epstein <jeeproject@yahoo.com>\nTO: [""<gmax1@mindspring.com>""]\nSUBJECT: Re...",0,0.0
3,0036be5386133888b4201acacd0d6178,"DATE: 2017-05-18T20:56:45.000Z\nFROM: Ryan Andrews <ryan.andrews@sf-email.sharefile.com>\nTO: [""<jeeproject@yahoo.co...",0,0.0
4,0037a69505ae78dfacfebd97260f5325,"DATE: 2014-11-26T19:03:47.000Z\nFROM: The New York Times <nytimes@e.newyorktimesinfo.com>\nTO: [""<jeeproject@yahoo.c...",0,0.0


## 5. Statistiche Descrittive

Analizziamo la distribuzione delle feature principali.

In [5]:
cols_to_show = ["combined_text_length", "redaction_count", "redaction_ratio", "recipient_count_estimate", "attachment_count"]
display(processed_sample[cols_to_show].describe().round(4))

,combined_text_length,redaction_count,redaction_ratio,recipient_count_estimate,attachment_count
count,355.0,355.0,355.0,355.0,355.0
mean,1647.8986,0.031,0.0001,1.1324,0.3662
std,2286.0387,0.2639,0.0009,0.6351,1.4423
min,141.0,0.0,0.0,1.0,0.0
25%,400.5,0.0,0.0,1.0,0.0
50%,870.0,0.0,0.0,1.0,0.0
75%,1947.5,0.0,0.0,1.0,0.0
max,17605.0,3.0,0.0116,8.0,14.0


## 6. Verifica Template Embeddings

Mostriamo un esempio di testo formattato con il template strutturato.

In [6]:
print("Esempio di combined_text (Templated):")
print("="*50)
print(processed_sample["combined_text"].iloc[0])
print("="*50)

Esempio di combined_text (Templated):
DATE: 2015-05-07T07:07:33.000Z
FROM: jeffrey E.  <jeevacation@gmail.com>
TO: ["Ehud Barak  <ehbarak1@gmail.com>"]
SUBJECT: Re: Title and subtitle

BODY:
prefer OUR COUNTRY , MY LIFE or MOMENTS


## 7. Controlli Finali

Verifichiamo che il dataset rispetti i vincoli del contratto.

In [7]:
required_columns = {
    "combined_text",
    "combined_text_length",
    "has_redaction",
    "redaction_count",
    "redaction_ratio",
    "recipient_count_estimate",
}
missing_columns = required_columns - set(processed_sample.columns)

assert processed_sample.shape[0] > 0, "Il dataset processato è vuoto."
assert not missing_columns, f"Colonne mancanti: {missing_columns}"

print("Tutti i controlli completati con successo.")

Tutti i controlli completati con successo.
